In [ ]:
import pandas as pd
import json
import time
from openai import OpenAI

# OpenRouter Config
OPENROUTER_API_KEY = "..."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL_NAME = "openai/gpt-5.4-mini"

# Load data
df = pd.read_csv('../preprocessing/preprocessing_output/data_judi_online_komentar_youtube_cleaned.csv')
print(f"Jumlah data: {len(df)}")

# Fungsi untuk memproses satu batch dengan OpenRouter
def label_batch_with_openrouter(batch_df):
    pesan_list = []
    for index, row in batch_df.iterrows():
        pesan_list.append({"id": index, "pesan": row['Pesan_Clean']})
    
    data_json_str = json.dumps(pesan_list, ensure_ascii=False)
    
    prompt = f"""
    Anda adalah asisten AI untuk mengklasifikasikan pesan live chat.
    Evaluasi setiap pesan dan berikan penilaian untuk 1 kategori:
    
    1. Kategori Toxic: Beri nilai "toxic" (jika ada kata kasar, makian, hinaan) atau "normal" (jika tidak ada).

    Ini adalah daftar pesannya:
    {data_json_str}

    KEMBALIKAN OUTPUT HANYA DALAM FORMAT JSON ARRAY SEPERTI INI:
    [
      {{"id": 0, "label_toxic": "normal"}},
      {{"id": 1, "label_toxic": "toxic"}},
      {{"id": 2, "label_toxic": "normal"}},
      {{"id": 3, "label_toxic": "toxic"}}
    ]
    JANGAN tambahkan teks penjelasan apapun, murni JSON valid.
    """
    
    try:
        # Request OpenRouter
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Anda adalah mesin pengklasifikasi teks yang hanya membalas dengan format JSON array valid."},
                {"role": "user", "content": prompt}
            ],
            extra_headers={
                "HTTP-Referer": "https://localhost", 
                "X-Title": "PelabelanDataset", 
            }
        )
        
        raw_text = response.choices[0].message.content.strip()
        
        # Bersihkan format markdown
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
            
        # Parse ke JSON Python
        hasil_json = json.loads(raw_text.strip())
        return hasil_json
        
    except Exception as e:
        print(f"Error pada batch: {e}")
        if 'raw_text' in locals():
            print(f"Raw response: {raw_text}")
        return None

BATCH_SIZE = 30
hasil_akhir_toxic = {}

print(f"Mulai pelabelan dengan model {MODEL_NAME}...")
for i in range(0, len(df), BATCH_SIZE):
    batch_df = df.iloc[i : i + BATCH_SIZE]
    
    hasil_batch = label_batch_with_openrouter(batch_df)
    
    if hasil_batch:
        for item in hasil_batch:
            if 'id' in item and 'label_toxic' in item:
                hasil_akhir_toxic[item['id']] = item['label_toxic']
    
    print(f"Berhasil memproses baris {i} sampai {i + len(batch_df) - 1}")
    time.sleep(2)

# Gabungkan hasil ke DataFrame
df['Label Toxic'] = df.index.map(hasil_akhir_toxic)
df['Label Toxic'] = df['Label Toxic'].fillna('error')

# Save CSV
df.to_csv('new_output/data_judi_online_komentar_youtube_cleaned_labeled.csv', index=False)
print("\nSelesai! File disimpan sebagai 'data_judi_online_komentar_youtube_cleaned_labeled.csv'")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())

Jumlah data: 8906
Mulai pelabelan dengan model openai/gpt-5.4-mini...
Berhasil memproses baris 0 sampai 29
Berhasil memproses baris 30 sampai 59
Berhasil memproses baris 60 sampai 89
Berhasil memproses baris 90 sampai 119
Berhasil memproses baris 120 sampai 149
Berhasil memproses baris 150 sampai 179
Berhasil memproses baris 180 sampai 209
Berhasil memproses baris 210 sampai 239
Berhasil memproses baris 240 sampai 269
Berhasil memproses baris 270 sampai 299
Berhasil memproses baris 300 sampai 329
Berhasil memproses baris 330 sampai 359
Berhasil memproses baris 360 sampai 389
Berhasil memproses baris 390 sampai 419
Berhasil memproses baris 420 sampai 449
Berhasil memproses baris 450 sampai 479
Berhasil memproses baris 480 sampai 509
Berhasil memproses baris 510 sampai 539
Berhasil memproses baris 540 sampai 569
Berhasil memproses baris 570 sampai 599
Berhasil memproses baris 600 sampai 629
Berhasil memproses baris 630 sampai 659
Berhasil memproses baris 660 sampai 689
Berhasil memproses